# **Step 1 - Paraphrasing Quality Experiments**

In [44]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

new_model_name = "humarin/chatgpt_paraphraser_on_T5_base"

new_tokenizer = AutoTokenizer.from_pretrained(new_model_name)

new_model = AutoModelForSeq2SeqLM.from_pretrained(
    new_model_name
).to(device)

print("Alternative model loaded successfully!")

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Alternative model loaded successfully!


In [45]:
def paraphrase_with_new_model(text):

    input_text = "paraphrase: " + text

    inputs = new_tokenizer(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)

    outputs = new_model.generate(
        **inputs,
        max_length=256,
        num_beams=5,
        num_return_sequences=3,
        do_sample=True,
        top_p=0.95,
        temperature=0.8,
        repetition_penalty=1.5
    )

    results = []

    for output in outputs:
        result = new_tokenizer.decode(
            output,
            skip_special_tokens=True
        )
        results.append(result)

    return results

In [46]:
original_text = """
Machine learning is a branch of artificial intelligence
that enables computers to learn from data without being
explicitly programmed. It is widely used in healthcare,
finance, education and many other industries.
"""

new_results = paraphrase_with_new_model(original_text)

for i, result in enumerate(new_results, 1):
    print(f"\nVersion {i}:")
    print(result)


Version 1:
Machine learning, a subset of artificial intelligence that allows computers to learn from data without programming, is commonly used in fields such as healthcare, finance, and education.

Version 2:
Machine learning, a type of artificial intelligence that allows computers to learn from data without programming, is commonly used in fields such as healthcare, finance, and education.

Version 3:
Machine learning, a type of artificial intelligence that allows computers to learn from data without programming, is commonly used in fields like healthcare, finance, and education.


# **Step 2 - Preprocessing**

In [47]:
# Import Libraries

import re
import nltk

from nltk.tokenize import sent_tokenize, word_tokenize

nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [48]:
# Preprocessing

def preprocess_text(text):

    # Remove unnecessary spaces
    text = re.sub(r'\s+', ' ', text)

    # Remove spaces before punctuation
    text = re.sub(r'\s+([.,!?;:])', r'\1', text)

    # Remove repeated punctuation
    text = re.sub(r'([.,!?])\1+', r'\1', text)

    # Remove leading and trailing spaces
    text = text.strip()

    return text

In [49]:
# Test Preprocessing

sample_text = """
    Machine learning   is transforming industries.

    It helps businesses automate tasks !!!


"""

cleaned_text = preprocess_text(sample_text)

print("Original Text:")
print(sample_text)

print("\nCleaned Text:")
print(cleaned_text)

Original Text:

    Machine learning   is transforming industries.

    It helps businesses automate tasks !!!




Cleaned Text:
Machine learning is transforming industries. It helps businesses automate tasks!


# **Step 3 - Grammar and Spelling Checker**

In [50]:
# Input Validation

def validate_input(text):

    if not isinstance(text, str):
        return False, "Input must be a string."

    if not text.strip():
        return False, "Please enter some text."

    if len(text.split()) < 5:
        return False, "Please enter at least five words."

    return True, "Input is valid."

In [51]:
test_text = "Machine learning improves productivity."

is_valid, message = validate_input(test_text)

print(message)

Please enter at least five words.


In [52]:
# Install Language Tool

!pip install language-tool-python

In [53]:
# Initialize the Grammar Checker

import language_tool_python

tool = language_tool_python.LanguageTool('en-US')

print("Grammar checker initialized successfully!")

Grammar checker initialized successfully!


In [54]:
# Create a Grammar Checking Function

def check_grammar(text):

    matches = tool.check(text)

    print("Grammar and Spelling Analysis")
    print("--------------------------------")

    if len(matches) == 0:
        print("No grammar or spelling issues detected.")

    else:
        print("Total issues found:", len(matches))

        for match in matches:

            incorrect_text = text[
                match.offset: match.offset + match.error_length
            ]

            print("\nIssue:", match.message)

            print("Incorrect text:", incorrect_text)

            print("Suggested correction:", match.replacements[:3])

    return matches

In [55]:
# Test the Grammar Checker

sample_text = """
Machine learning are transforming industries.
It help businesses automate repetitive task.
Artificial intelligence have changed the world.
"""

grammar_results = check_grammar(sample_text)

Grammar and Spelling Analysis
--------------------------------
Total issues found: 1

Issue: After ‘It’, use the third-person verb form “helps”.
Incorrect text: help
Suggested correction: ['helps']


In [56]:
# Test Automatic Correction

def correct_grammar(text):

    corrected_text = tool.correct(text)

    return corrected_text

In [57]:
corrected = correct_grammar(sample_text)

print("Original Text:")
print(sample_text)

print("\nCorrected Text:")
print(corrected)

Original Text:

Machine learning are transforming industries.
It help businesses automate repetitive task.
Artificial intelligence have changed the world.


Corrected Text:

Machine learning are transforming industries.
It helps businesses automate repetitive task.
Artificial intelligence have changed the world.



# **Step 4 - Semantic Similarity**

In [58]:
# Load the Model

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

similarity_model = SentenceTransformer(
    'sentence-transformers/all-MiniLM-L6-v2'
)

print("Semantic similarity model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Semantic similarity model loaded successfully!


In [59]:
# Create Similarity Function

def calculate_similarity(original, paraphrased):

    # Convert sentences into embeddings
    embeddings = similarity_model.encode(
        [original, paraphrased]
    )

    # Calculate cosine similarity
    similarity = cosine_similarity(
        [embeddings[0]],
        [embeddings[1]]
    )[0][0]

    return round(float(similarity), 4)

In [60]:
# Test Semantic Similarity

original = "Machine learning helps computers learn from data."

paraphrased = "Computers can identify patterns and improve their performance using data."

score = calculate_similarity(original, paraphrased)

print("Original:", original)
print("Paraphrased:", paraphrased)
print("Semantic Similarity Score:", score)
print("Similarity Percentage:", round(score * 100, 2), "%")

Original: Machine learning helps computers learn from data.
Paraphrased: Computers can identify patterns and improve their performance using data.
Semantic Similarity Score: 0.6278
Similarity Percentage: 62.78 %


In [61]:
# Compare 2 Different Paraphrases

original = "Machine learning helps computers learn from data."

version_1 = "Machine learning allows computers to learn from information."

version_2 = "The weather is very pleasant today."

print("Version 1 Similarity:",
      calculate_similarity(original, version_1))

print("Version 2 Similarity:",
      calculate_similarity(original, version_2))

Version 1 Similarity: 0.9488
Version 2 Similarity: -0.0151


# **Step 5 - BLEU and ROUGE evaluation**

In [62]:
# Import Libraries

import sacrebleu
from rouge_score import rouge_scorer

In [63]:
# Create Evaluation Function

def evaluate_bleu_rouge(reference, generated):

    # Calculate BLEU score
    bleu = sacrebleu.sentence_bleu(
        generated,
        [reference]
    )

    # Initialize ROUGE scorer
    scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rouge2', 'rougeL'],
        use_stemmer=True
    )

    # Calculate ROUGE scores
    rouge = scorer.score(
        reference,
        generated
    )

    print("BLEU Score:", round(bleu.score, 4))

    print("\nROUGE Scores:")

    for metric, score in rouge.items():

        print(
            metric,
            "Precision:", round(score.precision, 4),
            "Recall:", round(score.recall, 4),
            "F1:", round(score.fmeasure, 4)
        )

    return bleu, rouge

In [64]:
# Test BLEU & ROUGE

reference = "Machine learning helps computers learn from data."

generated = "Machine learning allows computers to learn from information."

bleu_result, rouge_result = evaluate_bleu_rouge(
    reference,
    generated
)

BLEU Score: 14.9237

ROUGE Scores:
rouge1 Precision: 0.625 Recall: 0.7143 F1: 0.6667
rouge2 Precision: 0.2857 Recall: 0.3333 F1: 0.3077
rougeL Precision: 0.625 Recall: 0.7143 F1: 0.6667


# **Step 6 - Fluency and Originality Analysis**

In [65]:
# Fluency Checking

def calculate_fluency(text):

    matches = tool.check(text)

    total_errors = len(matches)

    word_count = len(text.split())

    if word_count == 0:
        return 0

    error_rate = total_errors / word_count

    fluency_score = max(0, 1 - error_rate)

    return round(fluency_score, 4)

In [66]:
# Originality Analysis

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def calculate_originality(original, paraphrased):

    vectorizer = TfidfVectorizer()

    vectors = vectorizer.fit_transform(
        [original, paraphrased]
    )

    lexical_similarity = cosine_similarity(
        vectors[0],
        vectors[1]
    )[0][0]

    originality_score = 1 - lexical_similarity

    return round(float(originality_score), 4)

In [67]:
# Test Both Functions

original = "Machine learning helps computers learn from data."

paraphrased = "Machine learning allows computers to learn from information."

fluency = calculate_fluency(paraphrased)

originality = calculate_originality(
    original,
    paraphrased
)

print("Fluency Score:", fluency)
print("Originality Score:", originality)

Fluency Score: 1.0
Originality Score: 0.4944


In [68]:
# Combine All Evaluation Matrics

def evaluate_paraphrase(original, paraphrased, reference):

    semantic = calculate_similarity(
        original,
        paraphrased
    )

    fluency = calculate_fluency(
        paraphrased
    )

    originality = calculate_originality(
        original,
        paraphrased
    )

    bleu, rouge = evaluate_bleu_rouge(
        reference,
        paraphrased
    )

    print("\n========== FINAL EVALUATION ==========")

    print("Semantic Similarity:", semantic)
    print("Fluency Score:", fluency)
    print("Originality Score:", originality)
    print("BLEU Score:", round(bleu.score, 4))

    print("\nROUGE:")
    for metric, score in rouge.items():
        print(metric, round(score.fmeasure, 4))

    return {
        "semantic_similarity": semantic,
        "fluency": fluency,
        "originality": originality,
        "bleu": bleu.score,
        "rouge1": rouge["rouge1"].fmeasure,
        "rouge2": rouge["rouge2"].fmeasure,
        "rougeL": rouge["rougeL"].fmeasure
    }

# **Step 7 - Create AI-Powered Paraphrasing Tool**

In [69]:
# Create Paraphrasing Function

def generate_paraphrases(text):

    input_text = "paraphrase: " + text

    inputs = new_tokenizer(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)

    outputs = new_model.generate(
        **inputs,
        max_length=256,
        num_beams=5,
        num_return_sequences=3,
        do_sample=True,
        top_p=0.95,
        temperature=0.8,
        repetition_penalty=1.5
    )

    results = []

    for output in outputs:

        result = new_tokenizer.decode(
            output,
            skip_special_tokens=True
        )

        results.append(result)

    return results

In [70]:
def paraphrasing_application():

    print("=" * 55)
    print("       AI-POWERED PARAPHRASING TOOL")
    print("=" * 55)

    original_text = input("\nEnter your paragraph:\n")

    # Validate input
    is_valid, message = validate_input(original_text)

    if not is_valid:
        print("\nError:", message)
        return

    # Preprocess input
    cleaned_text = preprocess_text(original_text)

    print("\nGenerating paraphrases...")

    # Generate alternatives
    results = generate_paraphrases(cleaned_text)

    print("\nAVAILABLE PARAPHRASES")

    for i, result in enumerate(results, 1):
        print(f"\nVersion {i}:")
        print(result)

    # User selects a version
    try:
        choice = int(input("\nSelect a version (1-3): "))

        if choice not in [1, 2, 3]:
            print("Invalid selection.")
            return

    except ValueError:
        print("Please enter a valid number.")
        return

    selected_text = results[choice - 1]

    # Grammar correction
    corrected_text = correct_grammar(selected_text)

    # Evaluation
    print("\nSELECTED PARAPHRASE")
    print(corrected_text)

    print("\nGRAMMAR ANALYSIS")
    check_grammar(corrected_text)

    print("\nQUALITY EVALUATION")

    semantic = calculate_similarity(
        cleaned_text,
        corrected_text
    )

    fluency = calculate_fluency(corrected_text)

    originality = calculate_originality(
        cleaned_text,
        corrected_text
    )

    print("Semantic Similarity:", semantic)
    print("Fluency Score:", fluency)
    print("Originality Score:", originality)

    print("\n" + "=" * 55)
    print("          PARAPHRASING COMPLETED")
    print("=" * 55)

In [71]:
paraphrasing_application()

       AI-POWERED PARAPHRASING TOOL

Enter your paragraph:
RPA is a form of business process automation technology to build software robots to automate tasks and mimic the human interaction with software

Generating paraphrases...

AVAILABLE PARAPHRASES

Version 1:
The use of RPA, a type of business process automation technology, involves the creation of software robots that can automate tasks and imitate human interaction with software.

Version 2:
The use of RPA, a form of business process automation, involves the creation of software robots that can automate tasks and imitate human interaction with software.

Version 3:
The use of RPA, a type of business process automation, involves the creation of software robots that can automate tasks and imitate human interaction with software.

Select a version (1-3): 1

SELECTED PARAPHRASE
The use of RPA, a type of business process automation technology, involves the creation of software robots that can automate tasks and imitate human interac